# LFortran - 10 Hz Sine Wave

Created with Grok Build

**Kernel:** LFortran (`fortran`). Jupyter starts this notebook with `lfortran kernel -f {connection_file}`.

A one-second sine at 10 Hz, amplitude 1, sampled 1024 times. The sample rate is 1024 Hz, so each frequency bin is 1 Hz and the tone sits on bin 10. The magnitude there is 1, and the phase is -90 degrees, the phase of a sine. The [10 Hz FFTW3 page](../DFFT/Sine_Wave_10_Hz/sine_wave_10_hz.md) used 1000 Hz sampling and 1001 samples. This notebook uses 1024 samples, a power of two, which is the length `spectrum_of` accepts.

The printed table lists every bin whose magnitude is at least 0.05. The columns are frequency in hertz, magnitude, and phase in degrees. The time plot shows the first 0.2 seconds, two cycles. The spectrum plot runs out to 40 Hz. The phase plot marks the bins the table prints.

The shared modules are also written up on the [LFortran](lfortran__HEAD__.md) page. This notebook carries the same Fortran source in its first code cell.


## Kernel

The kernelspec stored in this notebook is `name: fortran`, `display_name: Fortran`, `language: fortran`. Jupyter uses that name to start LFortran with `lfortran kernel -f {connection_file}`. Both code cells are Fortran sent to that kernel. The first cell defines the modules. The second cell is top-level statements in the same session, so its `use` statements see `signals` and `spectrum`. The table under the second cell is the stdout from that run. Rebuilding the website publishes the saved stdout and does not start LFortran again.

## The module cell

The next cell is the first code cell. The LFortran kernel executes it. The source is `code/signals.f90`, a blank line, and `code/spectrum.f90`. The [LFortran](lfortran__HEAD__.md) page lists those same files. Running the cell defines modules `signals` and `spectrum` in this notebook.

`signals` is private by default and publishes `sine_tone`, `square_wave`, `two_sines`, and `switched_sine`. Kinds are `int32` and `real64` from `iso_fortran_env`. Sample times come from the private function `time_at`: sample `i`, counting from 1, is at `(i - 1) / sample_rate`.

`sine_tone` fills the array with `amplitude * sin(2π f t)`. `π` is `acos(-1.0_real64)`.

`square_wave` builds a 50% duty square between `+amplitude` and `-amplitude`. The fractional position in the period is `modulo(t * frequency, 1)`. `merge` selects `+amplitude` when that position is below one half, and `-amplitude` otherwise.

`two_sines` adds two sines. Each term has its own frequency and amplitude, and both use the same sample time.

`switched_sine` multiplies a carrier sine by a gate. For the first half of each gate period the sample is `amplitude * sin(2π f_carrier t)`. For the second half the sample is 0. The gate decision is also a `merge`.

`spectrum` publishes `spectrum_of`, `write_series`, and `print_peaks`. `spectrum_of` stops with `error stop` unless the length `N` is a power of two, tested with `N >= 2` and `iand(N, N - 1) == 0`. It packs the real samples into a `complex(real64)` array and calls the private in-place radix-2 FFT. That FFT bit-reverses the bins, then combines even and odd halves for block lengths 2, 4, and so on up to `N`. The twiddle angle is `-2π / length`, the same sign NumPy uses. After the transform, bin `k` has frequency `k * sample_rate / N`. The returned magnitude is `abs(X) / N` for DC and Nyquist, and `2 * abs(X) / N` for every other bin, so a sine of amplitude 1 peaks at magnitude 1. Phase is `atan2` of the imaginary and real parts, in degrees.

`write_series` opens the two paths passed by the next cell with `newunit` and writes `es24.16` columns. The time file is time and sample. The spectrum file is frequency, magnitude, and phase.

`print_peaks` prints the header `frequency_hz magnitude phase_deg` and calls the private `print_one_peak` for every bin. That routine formats one line, stores `magnitude >= threshold` in a logical, and returns before printing when the logical is false. A block-`if` on a real comparison inside a module procedure is miscompiled by this LFortran kernel, so the source uses the early return. This notebook calls the public `print_peaks` only.

This page calls `sine_tone`.


In [1]:
module signals
    use, intrinsic :: iso_fortran_env, only: int32, real64
    implicit none
    private
    public :: sine_tone, square_wave, two_sines, switched_sine

contains

    function sine_tone(n, sample_rate, frequency, amplitude) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: frequency
        real(real64), intent(in) :: amplitude
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: pi

        pi = acos(-1.0_real64)
        allocate(samples(n))
        do i = 1, n
            samples(i) = amplitude * sin(2.0_real64 * pi * frequency * time_at(i, sample_rate))
        end do
    end function sine_tone

    function square_wave(n, sample_rate, frequency, amplitude) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: frequency
        real(real64), intent(in) :: amplitude
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: position

        allocate(samples(n))
        do i = 1, n
            position = modulo(time_at(i, sample_rate) * frequency, 1.0_real64)
            samples(i) = merge(amplitude, -amplitude, position < 0.5_real64)
        end do
    end function square_wave

    function two_sines(n, sample_rate, frequency_a, amplitude_a, &
            frequency_b, amplitude_b) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: frequency_a
        real(real64), intent(in) :: amplitude_a
        real(real64), intent(in) :: frequency_b
        real(real64), intent(in) :: amplitude_b
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: pi, t

        pi = acos(-1.0_real64)
        allocate(samples(n))
        do i = 1, n
            t = time_at(i, sample_rate)
            samples(i) = amplitude_a * sin(2.0_real64 * pi * frequency_a * t) &
                + amplitude_b * sin(2.0_real64 * pi * frequency_b * t)
        end do
    end function two_sines

    function switched_sine(n, sample_rate, carrier, gate, amplitude) result(samples)
        integer(int32), intent(in) :: n
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: carrier
        real(real64), intent(in) :: gate
        real(real64), intent(in) :: amplitude
        real(real64), allocatable :: samples(:)
        integer(int32) :: i
        real(real64) :: pi, t, position

        pi = acos(-1.0_real64)
        allocate(samples(n))
        do i = 1, n
            t = time_at(i, sample_rate)
            position = modulo(t * gate, 1.0_real64)
            samples(i) = merge(amplitude * sin(2.0_real64 * pi * carrier * t), &
                0.0_real64, position < 0.5_real64)
        end do
    end function switched_sine

    pure real(real64) function time_at(index, sample_rate)
        integer(int32), intent(in) :: index
        real(real64), intent(in) :: sample_rate
        time_at = real(index - 1, real64) / sample_rate
    end function time_at

end module signals

module spectrum
    use, intrinsic :: iso_fortran_env, only: int32, real64
    implicit none
    private
    public :: spectrum_of, write_series, print_peaks

contains

    subroutine spectrum_of(samples, sample_rate, magnitude, frequency, phase)
        real(real64), intent(in) :: samples(:)
        real(real64), intent(in) :: sample_rate
        real(real64), allocatable, intent(out) :: magnitude(:)
        real(real64), allocatable, intent(out) :: frequency(:)
        real(real64), allocatable, intent(out) :: phase(:)
        complex(real64), allocatable :: bins(:)
        integer(int32) :: n, k, half
        real(real64) :: pi, scale

        n = size(samples, kind=int32)
        if (.not. is_power_of_two(n)) then
            error stop 'spectrum_of: length must be a power of two'
        end if

        allocate(bins(n))
        do k = 1, n
            bins(k) = cmplx(samples(k), 0.0_real64, real64)
        end do
        call fft_inplace(bins)

        half = n / 2
        allocate(magnitude(half + 1))
        allocate(frequency(half + 1))
        allocate(phase(half + 1))
        pi = acos(-1.0_real64)
        scale = real(n, real64)
        do k = 0, half
            frequency(k + 1) = real(k, real64) * sample_rate / scale
            if (k == 0 .or. k == half) then
                magnitude(k + 1) = abs(bins(k + 1)) / scale
            else
                magnitude(k + 1) = 2.0_real64 * abs(bins(k + 1)) / scale
            end if
            phase(k + 1) = atan2(aimag(bins(k + 1)), real(bins(k + 1))) * (180.0_real64 / pi)
        end do
    end subroutine spectrum_of

    subroutine write_series(time_path, spectrum_path, sample_rate, samples, &
            magnitude, frequency, phase)
        character(len=*), intent(in) :: time_path
        character(len=*), intent(in) :: spectrum_path
        real(real64), intent(in) :: sample_rate
        real(real64), intent(in) :: samples(:)
        real(real64), intent(in) :: magnitude(:)
        real(real64), intent(in) :: frequency(:)
        real(real64), intent(in) :: phase(:)
        integer(int32) :: unit, i

        open(newunit=unit, file=time_path, status='replace', action='write')
        do i = 1, size(samples, kind=int32)
            write(unit, '(es24.16, 1x, es24.16)') &
                real(i - 1, real64) / sample_rate, samples(i)
        end do
        close(unit)

        open(newunit=unit, file=spectrum_path, status='replace', action='write')
        do i = 1, size(magnitude, kind=int32)
            write(unit, '(es24.16, 1x, es24.16, 1x, es24.16)') &
                frequency(i), magnitude(i), phase(i)
        end do
        close(unit)
    end subroutine write_series

    subroutine print_peaks(frequency, magnitude, phase, threshold)
        real(real64), intent(in) :: frequency(:)
        real(real64), intent(in) :: magnitude(:)
        real(real64), intent(in) :: phase(:)
        real(real64), intent(in) :: threshold
        integer(int32) :: i, count

        count = size(magnitude, kind=int32)
        print '(a)', 'frequency_hz magnitude phase_deg'
        do i = 0, count - 1
            call print_one_peak(frequency(i + 1), magnitude(i + 1), phase(i + 1), threshold)
        end do
    end subroutine print_peaks

    ! A block-if on a real comparison is miscompiled by the LFortran kernel.
    ! The test is stored in a logical, and the branch uses that logical.
    subroutine print_one_peak(frequency, magnitude, phase, threshold)
        real(real64), intent(in) :: frequency
        real(real64), intent(in) :: magnitude
        real(real64), intent(in) :: phase
        real(real64), intent(in) :: threshold
        character(len=48) :: line
        logical :: keep

        keep = magnitude >= threshold
        write(line, '(f12.3, 1x, f12.6, 1x, f12.3)') frequency, magnitude, phase
        if (.not. keep) return
        print '(a)', trim(line)
    end subroutine print_one_peak

    subroutine fft_inplace(bins)
        complex(real64), intent(inout) :: bins(:)
        complex(real64) :: twiddle, twiddle_step, even, odd
        integer(int32) :: n, bits, i, reversed, length, half, start, k
        real(real64) :: angle

        n = size(bins, kind=int32)
        bits = 0
        length = 1
        do while (length < n)
            length = length * 2
            bits = bits + 1
        end do

        do i = 0, n - 1
            reversed = reverse_bits(i, bits)
            if (reversed > i) then
                even = bins(i + 1)
                bins(i + 1) = bins(reversed + 1)
                bins(reversed + 1) = even
            end if
        end do

        length = 2
        do while (length <= n)
            half = length / 2
            angle = -2.0_real64 * acos(-1.0_real64) / real(length, real64)
            twiddle_step = cmplx(cos(angle), sin(angle), real64)
            do start = 1, n, length
                twiddle = cmplx(1.0_real64, 0.0_real64, real64)
                do k = 0, half - 1
                    even = bins(start + k)
                    odd = twiddle * bins(start + k + half)
                    bins(start + k) = even + odd
                    bins(start + k + half) = even - odd
                    twiddle = twiddle * twiddle_step
                end do
            end do
            length = length * 2
        end do
    end subroutine fft_inplace

    pure integer(int32) function reverse_bits(value, bits) result(reversed)
        integer(int32), intent(in) :: value
        integer(int32), intent(in) :: bits
        integer(int32) :: bit, remaining

        reversed = 0
        remaining = value
        do bit = 1, bits
            reversed = reversed * 2_int32 + mod(remaining, 2_int32)
            remaining = remaining / 2_int32
        end do
    end function reverse_bits

    pure logical function is_power_of_two(n)
        integer(int32), intent(in) :: n
        is_power_of_two = n >= 2 .and. iand(n, n - 1_int32) == 0
    end function is_power_of_two

end module spectrum


## The example cell

The next cell is the second code cell, still on the LFortran kernel. It is top-level Fortran in the session where the modules are already defined, so there is no `program` / `end program` wrapper.

`use, intrinsic :: iso_fortran_env, only: real64` brings in double precision. `use signals` and `use spectrum` bring in the procedures from the cell above. Four allocatable `real64` arrays hold the samples and the magnitude, frequency, and phase columns.

```fortran
samples = sine_tone(1024, 1024.0_real64, 10.0_real64, 1.0_real64)
```

The arguments are the length, the sample rate in hertz, the tone in hertz, and the amplitude. The record is one second. Ten cycles fit in 1024 samples, so the tone lands on bin 10.

`spectrum_of` fills `magnitude`, `frequency`, and `phase` from those samples at a sample rate of 1024 Hz.

`write_series` writes `sine_10_hz_time.dat` and `sine_10_hz_spectrum.dat`. The Plots section below shows the command that turns those files into the figures.

`print_peaks` uses the threshold `0.05_real64`. The stdout saved under the cell is one data line: 10.000 Hz, magnitude 1.000000, phase -90.000 degrees.


In [2]:
use, intrinsic :: iso_fortran_env, only: real64
use signals
use spectrum

real(real64), allocatable :: samples(:)
real(real64), allocatable :: magnitude(:)
real(real64), allocatable :: frequency(:)
real(real64), allocatable :: phase(:)

samples = sine_tone(1024, 1024.0_real64, 10.0_real64, 1.0_real64)
call spectrum_of(samples, 1024.0_real64, magnitude, frequency, phase)
call write_series('sine_10_hz_time.dat', 'sine_10_hz_spectrum.dat', &
    1024.0_real64, samples, magnitude, frequency, phase)
call print_peaks(frequency, magnitude, phase, 0.05_real64)


frequency_hz magnitude phase_deg
      10.000     1.000000      -90.000


## Plots

The figures below were drawn from the text files written by the example cell. From `Sections/Computer_Programming/Fortran/LFortran` the command is:

```bash
python3 code/plot_spectrum.py
```

`code/plot_spectrum.py` reads the text files for all five examples on the [LFortran](lfortran__HEAD__.md) page. For this page it checks the time of sample `i` against `(i - 1) / 1024` and the Fortran magnitude and phase against `numpy.fft.rfft` with the `1/N` and `2/N` scaling. Magnitude must agree to `1e-6`. Phase on bins at or above magnitude 0.05 must agree to `1e-3` degrees. It then draws the first 0.2 seconds, and magnitude and phase through 40 Hz. The time series is blue, the magnitude is rust, and the phase is drawn as teal markers on a scale from -180 to 180 degrees. The magnitude PNG is also copied to `images/lfortran_dfft.png`, the image on the LFortran card of the [Fortran](../fortran__HEAD__.md) page. After the check passes, the script deletes the `.dat` files. They are scratch and are not part of the site.

This is the script the command runs:

```{literalinclude} code/plot_spectrum.py
:language: python
```

:::{figure} images/sine_10_hz_time.png
:class: clickable-figure

The first 0.2 seconds of a 10 Hz sine sampled at 1024 Hz.
:::

:::{figure} images/sine_10_hz_spectrum.png
:class: clickable-figure

Magnitude spectrum of the 10 Hz sine. The peak is at 10 Hz and its magnitude is 1.
:::

:::{figure} images/sine_10_hz_phase.png
:class: clickable-figure

Phase of the bins at or above magnitude 0.05. The 10 Hz line is at -90 degrees.
:::

## Conclusion

The magnitude plot has one line, at 10 Hz, and its height is 1. That height is the amplitude of the sine. The factor of $2/N$ on every bin except DC and Nyquist folds the negative-frequency half of a real sine into the positive bin, so the peak reads as the amplitude. The phase plot marks that same bin at -90 degrees, the phase of a sine. Bins below magnitude 0.05 are left off the phase plot, the same cutoff as the printed table. The time plot is the first two cycles of that sine.

